In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

from students.kolozov.lesson2 import Exercise


def normalize_data(X_train, X_val, X_test):
    # Нормализация на основе статистик тренировочной выборки.
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0

    X_train_norm = (X_train - mean) / std
    X_val_norm = (X_val - mean) / std
    X_test_norm = (X_test - mean) / std

    return X_train_norm, X_val_norm, X_test_norm


def evaluate_model(lr, batch_size, X_train, X_val, y_train, y_val, n_iter, seed):
    # Создаем модель и обучаем
    rng = np.random.default_rng(seed)
    model = Exercise.create_logistic_model(num_features=X_train.shape[1], rng=rng)
    Exercise.fit(model, X_train, y_train, lr=lr, n_iter=n_iter, batch_size=batch_size)

    # Оцениваем на валидации
    auroc = model.metric(X_val, y_val, metric="AUROC")

    return auroc, model


def main():
    # Загрузка данных

    iris = load_iris()
    X = iris.data
    y = iris.target
    y_binary = (y < 1).astype(int)  # 1 если класс 0, 0 если класс 1 или 2

    print(f"Всего образцов: {len(X)}")
    print(f"Класс 0: {np.sum(y_binary == 1)} шт, Класс 1 + 2: {np.sum(y_binary == 0)} шт")

    # Фиксированное разделение 60/20/20
    X_train, X_temp, y_train, y_temp = train_test_split(X, y_binary, test_size=0.4, random_state=42, stratify=y_binary)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    # Нормализация
    X_train_norm, X_val_norm, X_test_norm = normalize_data(X_train, X_val, X_test)

    # Подбор гиперпараметров
    n_iter = 25
    seeds = [42, 123, 456, 2026]

    lr_values = [0.001, 0.003, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0]
    batch_size_values = [2, 4, 8, 16, 32, None]

    best_config = {"lr": 0, "batch_size": None, "auroc_mean": 0.0, "auroc_std": 1.0}
    results = []

    for lr in lr_values:
        for batch_size in batch_size_values:
            auroc_scores = []

            for seed in seeds:
                auroc, _ = evaluate_model(lr, batch_size, X_train_norm, X_val_norm, y_train, y_val, n_iter, seed)
                auroc_scores.append(auroc)

            auroc_mean = np.mean(auroc_scores)
            auroc_std = np.std(auroc_scores)

            results.append((lr, batch_size, auroc_mean, auroc_std))

            if auroc_mean > best_config["auroc_mean"] or (
                auroc_mean == best_config["auroc_mean"] and auroc_std < best_config["auroc_std"]
            ):
                best_config.update(
                    {"lr": lr, "batch_size": batch_size, "auroc_mean": auroc_mean, "auroc_std": auroc_std}
                )

    results.sort(key=lambda x: (-x[2], x[3]))

    print("\n" + "=" * 80)
    print(f"{'ТОП КОМБИНАЦИЙ'}")
    print("=" * 80)
    print(f"{'№':<3} {'lr':<10} {'batch_size':<12} {'AUROC mean':<12} {'AUROC std':<10}")
    print("-" * 80)

    for i, (lr, bs, mean, std) in enumerate(results[:7], 1):
        bs_str = str(bs) if bs is not None else "None"
        print(f"{i:<3} {lr:<10} {bs_str:<12} {mean:.4f}       ±{std:.4f}")

    print("=" * 80)
    print("\n" + "=" * 80)
    print("ЛУЧШАЯ КОНФИГУРАЦИЯ")
    print("=" * 80)
    print(f"Learning Rate:      {best_config['lr']}")
    print(f"Batch Size:         {best_config['batch_size']}")
    print(f"Val AUROC:          {best_config['auroc_mean']:.4f} ± {best_config['auroc_std']:.4f}")
    print(f"Epochs:             {n_iter}")

    # Финальная проверка на тестовом наборе
    final_rng = np.random.default_rng(42)
    final_model = Exercise.create_logistic_model(num_features=X_train_norm.shape[1], rng=final_rng)
    Exercise.fit(
        final_model, X_train_norm, y_train, lr=best_config["lr"], n_iter=n_iter, batch_size=best_config["batch_size"]
    )

    test_auroc = final_model.metric(X_test_norm, y_test, metric="AUROC")
    test_accuracy = final_model.metric(X_test_norm, y_test, metric="accuracy")

    print(f"\nTest AUROC:         {test_auroc:.4f}")
    print(f"Test Accuracy:      {test_accuracy:.4f}")
    print(f"Gap (Val - Test):   {best_config['auroc_mean'] - test_auroc:.4f}")

    return best_config


if __name__ == "__main__":
    main()